# 01 — Load and explore

This notebook validates the downloaded Kaggle files, creates the local DuckDB
database, and summarizes the resulting schema. It can be launched from the project
root or the `notebooks` directory.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "sql" / "create_tables.sql").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing sql/create_tables.sql")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATABASE_PATH = PROJECT_ROOT / "instacart.duckdb"
PROJECT_ROOT

PosixPath('/Users/sebastiankalita/Documents/DE_portfolio/sql-project/instacart-sql-analysis')

## Validate source files

In [2]:
required_files = [
    "aisles.csv",
    "departments.csv",
    "orders.csv",
    "products.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
]
missing_files = [name for name in required_files if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        "Missing files in data/raw: " + ", ".join(missing_files)
    )

pd.DataFrame({"source_file": required_files, "available": True})

,source_file,available
0,aisles.csv,True
1,departments.csv,True
2,orders.csv,True
3,products.csv,True
4,order_products__prior.csv,True
5,order_products__train.csv,True


## Build the DuckDB tables

In [3]:
create_tables_sql = (PROJECT_ROOT / "sql" / "create_tables.sql").read_text()
absolute_create_sql = create_tables_sql.replace(
    "'data/raw/", f"'{DATA_DIR.as_posix()}/"
)

with duckdb.connect(str(DATABASE_PATH)) as connection:
    connection.execute(absolute_create_sql)

print(f"Database created at {DATABASE_PATH}")

Database created at /Users/sebastiankalita/Documents/DE_portfolio/sql-project/instacart-sql-analysis/instacart.duckdb


## Inspect table sizes

In [4]:
with duckdb.connect(str(DATABASE_PATH), read_only=True) as connection:
    tables = connection.execute(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'main'
        ORDER BY table_name
        """
    ).df()
    table_counts = pd.DataFrame(
        [
            {
                "table_name": table_name,
                "row_count": connection.execute(
                    f'SELECT COUNT(*) FROM "{table_name}"'
                ).fetchone()[0],
            }
            for table_name in tables["table_name"]
        ]
    )

table_counts

,table_name,row_count
0,aisles,134
1,departments,21
2,order_products,33819106
3,order_products_prior,32434489
4,order_products_train,1384617
5,orders,3421083
6,products,49688


## Inspect the schema

In [5]:
with duckdb.connect(str(DATABASE_PATH), read_only=True) as connection:
    schema = connection.execute(
        """
        SELECT table_name, column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'main'
        ORDER BY table_name, ordinal_position
        """
    ).df()

schema

,table_name,column_name,data_type
0,aisles,aisle_id,BIGINT
1,aisles,aisle,VARCHAR
2,departments,department_id,BIGINT
3,departments,department,VARCHAR
4,order_products,order_id,BIGINT
5,order_products,product_id,BIGINT
6,order_products,add_to_cart_order,BIGINT
7,order_products,reordered,BIGINT
8,order_products_prior,order_id,BIGINT
9,order_products_prior,product_id,BIGINT
